# Swin Transformer 从零实现：货架划痕方向识别

## 面试问题

面试时我会先说：Swin 先把图像切成 patch token，再只在局部窗口内做多头自注意力，把全局二次复杂度降为窗口内复杂度。连续两层不能一直使用同一窗口，否则跨窗口相邻 token 永远无法通信；第二层通过循环位移形成 shifted windows。循环位移产生的图像边界拼接必须用 attention mask 隔离，否则左边缘会错误看见右边缘。相对位置偏置让同一窗口内的注意力知道二维相对位移。Patch Merging 用于分层降采样，本教学模型为突出窗口机制而不做多 stage。下面在同一批货架灰度图上比较全局均值基线，手写窗口划分、相对偏置、W-MSA/SW-MSA，并直接测量跨窗信息传播。

## 真实案例

案例包含八张 8×8 离线货架巡检图：四张是竖向划痕、四张是横向划痕，划痕位置不同但亮像素总数相同。字符图直接显示输入；真实系统会使用更高分辨率、颜色、复杂背景和独立验证集。

本实验是用于解释机制的确定性小样本，所有指标均标记为“教学实验”，不能外推为线上收益。

In [1]:
import math  # 导入平方根用于缩放点积注意力。
import torch  # 导入 PyTorch 以实现窗口注意力和训练。
from torch import nn  # 导入神经网络基础模块。
import torch.nn.functional as F  # 导入激活与交叉熵等基础算子。
torch.manual_seed(46)  # 固定随机种子以复现实验输出。
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小实验耗时。
sample_names = ["货架-A1", "货架-A2", "货架-A3", "货架-A4", "货架-B1", "货架-B2", "货架-B3", "货架-B4"]  # 定义八张脱敏巡检图编号。
images = torch.full((8, 1, 8, 8), 0.05, dtype=torch.float32)  # 创建带轻微暗背景的八张灰度图。
vertical_columns = [1, 2, 4, 5]  # 定义四条竖向划痕所在列。
horizontal_rows = [1, 2, 4, 5]  # 定义四条横向划痕所在行。
for index, column in enumerate(vertical_columns):  # 逐张写入贯穿货架的竖向亮划痕。
    images[index, 0, :, column] = 1.0  # 把当前划痕列的八个像素设为高亮。
for offset, row in enumerate(horizontal_rows):  # 逐张写入贯穿货架的横向亮划痕。
    images[4 + offset, 0, row, :] = 1.0  # 把当前划痕行的八个像素设为高亮。
labels = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)  # 用零表示竖划痕并用一表示横划痕。
print("输入字符图：# 表示高亮划痕，. 表示货架背景")  # 说明字符图中两种符号的含义。
for index, sample_name in enumerate(sample_names):  # 逐张打印真实可读的二维输入。
    print(f"\n{sample_name}  标签={'竖向' if labels[index] == 0 else '横向'}")  # 输出当前样本名称和人工标签。
    for row in images[index, 0]:  # 逐行把像素转成字符以显示空间结构。
        print("".join("#" if value > 0.5 else "." for value in row.tolist()))  # 输出当前图像行的八个字符。
print(f"输入张量形状={tuple(images.shape)}，每张亮像素数={[(images[index] > 0.5).sum().item() for index in range(len(images))]}")  # 展示形状与受控亮度统计。

输入字符图：# 表示高亮划痕，. 表示货架背景

货架-A1  标签=竖向
.#......
.#......
.#......
.#......
.#......
.#......
.#......
.#......

货架-A2  标签=竖向
..#.....
..#.....
..#.....
..#.....
..#.....
..#.....
..#.....
..#.....

货架-A3  标签=竖向
....#...
....#...
....#...
....#...
....#...
....#...
....#...
....#...

货架-A4  标签=竖向
.....#..
.....#..
.....#..
.....#..
.....#..
.....#..
.....#..
.....#..

货架-B1  标签=横向
........
########
........
........
........
........
........
........

货架-B2  标签=横向
........
........
########
........
........
........
........
........

货架-B3  标签=横向
........
........
........
........
########
........
........
........

货架-B4  标签=横向
........
........
........
........
........
########
........
........
输入张量形状=(8, 1, 8, 8)，每张亮像素数=[8, 8, 8, 8, 8, 8, 8, 8]


## 基线：只看全图平均亮度

八张图都有八个亮像素，因此全局均值完全丢失方向信息。基线统一预测竖向，并与 Swin 在相同八张图、相同准确率指标上比较。

In [2]:
mean_intensity = images.mean(dim=(1, 2, 3))  # 计算每张巡检图的全局平均亮度。
baseline_predictions = torch.zeros_like(labels)  # 在完全相同均值下稳定预测训练集中首个类别。
baseline_accuracy = (baseline_predictions == labels).float().mean().item()  # 计算全局亮度基线准确率。
print("样本      平均亮度  基线预测  真实标签")  # 打印逐样本基线结果表头。
for index, sample_name in enumerate(sample_names):  # 逐张展示均值特征为何无法区分方向。
    print(f"{sample_name}  {mean_intensity[index]:.4f}    {baseline_predictions[index].item()}         {labels[index].item()}")  # 输出当前样本的均值、预测和标签。
print(f"全局平均亮度基线准确率={baseline_accuracy:.1%}")  # 汇总同数据上的基线表现。

样本      平均亮度  基线预测  真实标签
货架-A1  0.1687    0         0
货架-A2  0.1687    0         0
货架-A3  0.1687    0         0
货架-A4  0.1687    0         0
货架-B1  0.1688    0         1
货架-B2  0.1688    0         1
货架-B3  0.1688    0         1
货架-B4  0.1688    0         1
全局平均亮度基线准确率=50.0%


## 手写核心：窗口划分、相对位置偏置和 shifted-window mask

`window_partition` 与 `window_reverse` 明确完成二维 token 和窗口批次之间的变换；`WindowAttention` 手算 Q/K/V、相对位置索引和 mask；`SwinBlock` 再负责循环位移、逆位移和残差前馈。

In [3]:
def window_partition(tokens, window_size):  # 把二维 token 网格切成互不重叠的小窗口。
    batch_size, height, width, channels = tokens.shape  # 读取 token 网格的四个维度。
    reshaped = tokens.view(batch_size, height // window_size, window_size, width // window_size, window_size, channels)  # 拆出窗口行列和窗口内部行列。
    windows = reshaped.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size * window_size, channels)  # 把每个窗口整理成 token 序列。
    return windows  # 返回形状为窗口批次、窗口 token 数、通道数的张量。
def window_reverse(windows, window_size, height, width, batch_size):  # 把窗口序列恢复成原二维 token 网格。
    reshaped = windows.view(batch_size, height // window_size, width // window_size, window_size, window_size, -1)  # 恢复窗口行列与内部行列。
    tokens = reshaped.permute(0, 1, 3, 2, 4, 5).contiguous().view(batch_size, height, width, -1)  # 交错窗口维和内部维以还原空间布局。
    return tokens  # 返回恢复后的二维 token 网格。
def build_shift_mask(height, width, window_size, shift_size, device):  # 构造阻止循环边界错误相连的 shifted-window mask。
    region_ids = torch.zeros((1, height, width, 1), device=device)  # 创建每个空间位置所属原区域的编号图。
    height_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))  # 按窗口和位移切分高度区域。
    width_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))  # 按窗口和位移切分宽度区域。
    region_number = 0  # 从零开始给九个边界区域编号。
    for height_slice in height_slices:  # 遍历三段高度区域。
        for width_slice in width_slices:  # 遍历三段宽度区域。
            region_ids[:, height_slice, width_slice, :] = region_number  # 为当前二维区域写入唯一编号。
            region_number += 1  # 递增编号供下一区域使用。
    mask_windows = window_partition(region_ids, window_size).squeeze(-1)  # 把区域编号按 shifted window 的布局切窗。
    pair_difference = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)  # 比较每个窗口内任意两个 token 的区域编号。
    attention_mask = torch.where(pair_difference == 0, torch.tensor(0.0, device=device), torch.tensor(-100.0, device=device))  # 同区域允许注意而跨循环边界强力屏蔽。
    return attention_mask  # 返回每个窗口对应的成对加性 mask。
class WindowAttention(nn.Module):  # 定义带二维相对位置偏置的手写窗口多头注意力。
    def __init__(self, model_dim, head_count, window_size):  # 根据隐藏维度、头数和窗口宽度创建参数。
        super().__init__()  # 初始化父类以注册全部参数。
        self.head_count = head_count  # 保存注意力头数供张量重排。
        self.head_dim = model_dim // head_count  # 计算每个注意力头的维度。
        self.window_size = window_size  # 保存方形窗口的边长。
        self.qkv = nn.Linear(model_dim, model_dim * 3, bias=False)  # 一次线性投影得到 Query、Key 和 Value。
        self.output = nn.Linear(model_dim, model_dim)  # 创建多头结果的输出投影。
        relative_count = (2 * window_size - 1) ** 2  # 计算二维相对位移的可能数量。
        self.relative_bias_table = nn.Parameter(torch.zeros(relative_count, head_count))  # 为每种相对位移和头学习偏置。
        coordinates = torch.stack(torch.meshgrid(torch.arange(window_size), torch.arange(window_size), indexing="ij"))  # 创建窗口内二维坐标网格。
        flattened = coordinates.flatten(1)  # 把二维坐标拉平成窗口 token 序列。
        relative = flattened[:, :, None] - flattened[:, None, :]  # 计算所有 token 对的二维相对坐标。
        relative = relative.permute(1, 2, 0).contiguous()  # 把坐标差维移动到最后便于编码。
        relative[:, :, 0] += window_size - 1  # 把可能为负的行位移平移到非负范围。
        relative[:, :, 1] += window_size - 1  # 把可能为负的列位移平移到非负范围。
        relative[:, :, 0] *= 2 * window_size - 1  # 给行位移乘基数以生成唯一索引。
        relative_index = relative.sum(dim=-1)  # 合并行列位移得到偏置表索引。
        self.register_buffer("relative_index", relative_index)  # 注册无需训练但随模型保存的位置索引。
    def forward(self, windows, attention_mask=None):  # 对窗口批次执行手写多头自注意力。
        window_batch, token_count, model_dim = windows.shape  # 读取窗口批次、token 数与隐藏维度。
        qkv = self.qkv(windows).view(window_batch, token_count, 3, self.head_count, self.head_dim).permute(2, 0, 3, 1, 4)  # 投影并拆出 Q/K/V 与多头维度。
        query, key, value = qkv[0], qkv[1], qkv[2]  # 分别读取 Query、Key 和 Value。
        scores = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算缩放点积注意力分数。
        relative_bias = self.relative_bias_table[self.relative_index.reshape(-1)].view(token_count, token_count, self.head_count).permute(2, 0, 1)  # 查表恢复每个头的二维相对位置偏置。
        scores = scores + relative_bias.unsqueeze(0)  # 给所有窗口加入共享相对位置偏置。
        if attention_mask is not None:  # shifted windows 需要额外隔离循环拼接区域。
            repeated_mask = attention_mask.repeat(window_batch // attention_mask.shape[0], 1, 1)  # 为批次中的每张图复制空间窗口 mask。
            scores = scores + repeated_mask.unsqueeze(1)  # 把加性 mask 广播到所有注意力头。
        attention = torch.softmax(scores, dim=-1)  # 在每个窗口的可见 token 上归一化权重。
        context = attention @ value  # 用注意力权重聚合窗口内 Value。
        merged = context.transpose(1, 2).contiguous().view(window_batch, token_count, model_dim)  # 把多个头拼回隐藏维度。
        return self.output(merged), attention  # 返回窗口上下文与可解释注意力矩阵。
class SwinBlock(nn.Module):  # 定义包含 W-MSA 或 SW-MSA 的手写 Swin Block。
    def __init__(self, model_dim, head_count, window_size, shift_size):  # 创建窗口注意力、归一化和前馈层。
        super().__init__()  # 初始化父类以注册所有子模块。
        self.window_size = window_size  # 保存窗口边长。
        self.shift_size = shift_size  # 保存循环位移距离。
        self.norm_one = nn.LayerNorm(model_dim)  # 创建窗口注意力前的归一化。
        self.attention = WindowAttention(model_dim, head_count, window_size)  # 创建手写窗口注意力模块。
        self.norm_two = nn.LayerNorm(model_dim)  # 创建前馈网络前的归一化。
        self.feedforward_up = nn.Linear(model_dim, model_dim * 2)  # 把通道扩展两倍形成前馈隐藏层。
        self.feedforward_down = nn.Linear(model_dim * 2, model_dim)  # 把前馈表示映射回模型维度。
    def forward(self, tokens):  # 对二维 token 网格执行一个完整 Swin Block。
        batch_size, height, width, channels = tokens.shape  # 读取二维 token 网格形状。
        normalized = self.norm_one(tokens)  # 对每个 token 的通道执行归一化。
        shifted = torch.roll(normalized, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2)) if self.shift_size > 0 else normalized  # 在 SW-MSA 层循环左上位移 token 网格。
        attention_mask = build_shift_mask(height, width, self.window_size, self.shift_size, tokens.device) if self.shift_size > 0 else None  # 仅为 shifted windows 构造边界 mask。
        windows = window_partition(shifted, self.window_size)  # 把位移后的 token 网格切成局部窗口。
        attended_windows, attention = self.attention(windows, attention_mask)  # 在每个窗口内计算多头注意力。
        attended_grid = window_reverse(attended_windows, self.window_size, height, width, batch_size)  # 把注意力窗口恢复成二维网格。
        restored = torch.roll(attended_grid, shifts=(self.shift_size, self.shift_size), dims=(1, 2)) if self.shift_size > 0 else attended_grid  # 逆位移恢复原空间坐标。
        hidden = tokens + restored  # 添加窗口注意力残差连接。
        feedforward = self.feedforward_down(F.gelu(self.feedforward_up(self.norm_two(hidden))))  # 计算逐 token 的两层 GELU 前馈网络。
        return hidden + feedforward, attention  # 返回块输出和窗口注意力矩阵。
class TinySwin(nn.Module):  # 定义用于划痕方向分类的两块教学版 Swin。
    def __init__(self):  # 创建 patch 投影、W-MSA、SW-MSA 和分类头。
        super().__init__()  # 初始化父类以注册所有参数。
        self.patch_projection = nn.Linear(4, 16)  # 把每个 2×2 灰度 patch 投影到十六维。
        self.regular_block = SwinBlock(16, 2, window_size=2, shift_size=0)  # 第一块在固定 2×2 token 窗口内注意。
        self.shifted_block = SwinBlock(16, 2, window_size=2, shift_size=1)  # 第二块位移一个 token 连接相邻窗口。
        self.final_norm = nn.LayerNorm(16)  # 创建全局池化前的通道归一化。
        self.classifier = nn.Linear(16, 2)  # 输出竖向和横向两个类别 logits。
    def patchify(self, image_batch):  # 手工把 8×8 图像切成 4×4 个非重叠 patch。
        batch_size = image_batch.shape[0]  # 读取当前图像批次大小。
        patches = image_batch.view(batch_size, 1, 4, 2, 4, 2).permute(0, 2, 4, 1, 3, 5).contiguous().view(batch_size, 4, 4, 4)  # 整理每个 2×2 patch 的四个像素。
        return patches  # 返回四行四列的 patch 特征网格。
    def forward(self, image_batch):  # 完成 patch embedding 和两种窗口注意力分类。
        raw_patches = self.patchify(image_batch)  # 把输入图像转为十六个可解释 patch。
        tokens = self.patch_projection(raw_patches)  # 把四像素 patch 投影成隐藏 token。
        regular_tokens, regular_attention = self.regular_block(tokens)  # 运行固定窗口注意力块。
        shifted_tokens, shifted_attention = self.shifted_block(regular_tokens)  # 运行 shifted-window 注意力块。
        pooled = self.final_norm(shifted_tokens).mean(dim=(1, 2))  # 对全部空间 token 做全局平均池化。
        logits = self.classifier(pooled)  # 输出划痕方向分类 logits。
        return logits, raw_patches, regular_tokens, shifted_tokens, regular_attention, shifted_attention  # 返回预测和全部关键中间量。
swin = TinySwin()  # 实例化手写窗口 Transformer。
print(swin)  # 展示网络确实由自写 Swin Block 构成。
print(f"可训练参数量={sum(parameter.numel() for parameter in swin.parameters())}")  # 输出教学模型参数规模。

TinySwin(
  (patch_projection): Linear(in_features=4, out_features=16, bias=True)
  (regular_block): SwinBlock(
    (norm_one): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (attention): WindowAttention(
      (qkv): Linear(in_features=16, out_features=48, bias=False)
      (output): Linear(in_features=16, out_features=16, bias=True)
    )
    (norm_two): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (feedforward_up): Linear(in_features=16, out_features=32, bias=True)
    (feedforward_down): Linear(in_features=32, out_features=16, bias=True)
  )
  (shifted_block): SwinBlock(
    (norm_one): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (attention): WindowAttention(
      (qkv): Linear(in_features=16, out_features=48, bias=False)
      (output): Linear(in_features=16, out_features=16, bias=True)
    )
    (norm_two): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (feedforward_up): Linear(in_features=16, out_features=32, bias=True)
    (feedforw

In [4]:
optimizer = torch.optim.Adam(swin.parameters(), lr=0.02)  # 创建优化器学习划痕方向模式。
loss_trace = []  # 保存真实训练损失轨迹。
first_gradient_norm = 0.0  # 预留首轮 QKV 梯度范数。
for epoch in range(251):  # 在八张教学图上执行二百五十一次参数更新。
    optimizer.zero_grad()  # 清空上一轮累计梯度。
    logits, raw_patches, regular_tokens, shifted_tokens, regular_attention, shifted_attention = swin(images)  # 运行完整手写 Swin 前向传播。
    loss = F.cross_entropy(logits, labels)  # 计算竖向与横向分类交叉熵。
    loss.backward()  # 反向传播到 patch 投影和窗口注意力参数。
    if epoch == 0:  # 首轮记录实际 QKV 梯度规模。
        first_gradient_norm = swin.regular_block.attention.qkv.weight.grad.norm().item()  # 读取固定窗口 QKV 投影梯度范数。
    optimizer.step()  # 根据当前梯度更新全部模型参数。
    loss_trace.append(loss.item())  # 保存当前轮损失用于收敛对照。
    if epoch in [0, 25, 100, 250]:  # 选择关键轮次输出真实训练轨迹。
        current_accuracy = (logits.argmax(dim=1) == labels).float().mean().item()  # 计算当前训练准确率。
        print(f"epoch={epoch:03d} loss={loss.item():.4f} accuracy={current_accuracy:.1%}")  # 输出损失和准确率变化。
swin.eval()  # 切换到评估模式获得稳定预测。
with torch.no_grad():  # 关闭评估阶段的梯度记录。
    final_logits, raw_patches, regular_tokens, shifted_tokens, regular_attention, shifted_attention = swin(images)  # 重新计算最终预测和中间张量。
probabilities = torch.softmax(final_logits, dim=1)[:, 1]  # 提取横向划痕类别概率。
predictions = final_logits.argmax(dim=1)  # 选择概率最大的划痕类别。
swin_accuracy = (predictions == labels).float().mean().item()  # 计算 Swin 在同一批图上的准确率。
first_window_attention = regular_attention[0].mean(dim=0)  # 对首张图首窗口的两个头取平均。
print(f"首轮 QKV 梯度范数={first_gradient_norm:.6f}")  # 输出非零梯度证明窗口注意力被真实训练。
print(f"patch 网格形状={tuple(raw_patches.shape)}，W-MSA 输出={tuple(regular_tokens.shape)}，SW-MSA 输出={tuple(shifted_tokens.shape)}")  # 展示空间 token 在各阶段的形状。
print("首张图第一个窗口的平均注意力矩阵：")  # 标记即将展示的四乘四注意力。
print(first_window_attention.round(decimals=3))  # 直接输出窗口内每个 token 对的真实权重。

epoch=000 loss=0.7447 accuracy=50.0%


epoch=025 loss=0.6869 accuracy=75.0%


epoch=100 loss=0.0007 accuracy=100.0%


epoch=250 loss=0.0001 accuracy=100.0%
首轮 QKV 梯度范数=0.177784
patch 网格形状=(8, 4, 4, 4)，W-MSA 输出=(8, 4, 4, 16)，SW-MSA 输出=(8, 4, 4, 16)
首张图第一个窗口的平均注意力矩阵：
tensor([[0.2660, 0.1800, 0.2400, 0.3130],
        [0.2860, 0.2070, 0.2170, 0.2900],
        [0.2770, 0.3230, 0.2290, 0.1710],
        [0.2520, 0.2980, 0.2510, 0.1990]])


## 结果解读

全局均值完全相同，而 patch token 保留了亮线在行或列上的排列。注意力矩阵每行之和为 1；逐样本概率则展示模型是否只记住某一个位置。

In [5]:
print("样本      真实方向  基线  横向概率  Swin预测")  # 打印逐图分类结果表头。
for index, sample_name in enumerate(sample_names):  # 遍历八张图展示同数据预测对照。
    true_name = "横向" if labels[index] == 1 else "竖向"  # 把整数标签转成可读方向。
    predicted_name = "横向" if predictions[index] == 1 else "竖向"  # 把模型类别转成可读方向。
    print(f"{sample_name}  {true_name:<4}     {baseline_predictions[index].item()}     {probabilities[index]:.3f}     {predicted_name}")  # 输出当前样本的概率和预测。
print(f"同数据准确率：均值基线={baseline_accuracy:.1%}，手写 Swin={swin_accuracy:.1%}")  # 汇总两个方案的同口径指标。

样本      真实方向  基线  横向概率  Swin预测
货架-A1  竖向       0     0.000     竖向
货架-A2  竖向       0     0.000     竖向
货架-A3  竖向       0     0.000     竖向
货架-A4  竖向       0     0.000     竖向
货架-B1  横向       0     1.000     横向
货架-B2  横向       0     1.000     横向
货架-B3  横向       0     1.000     横向
货架-B4  横向       0     1.000     横向
同数据准确率：均值基线=50.0%，手写 Swin=100.0%


## 失败案例：只堆固定窗口导致跨窗相邻点无法通信

在 4×4 token 网格中，位置 `(1,1)` 与 `(1,2)` 空间相邻，却属于两个固定窗口。只跑 W-MSA 时修改右侧 token 不影响左侧输出；再跑 SW-MSA 后，位移窗口把两者放入同一局部上下文。

In [6]:
def uniform_window_mix(tokens, shift_size):  # 用均匀注意力直接验证窗口拓扑而不受训练权重饱和影响。
    shifted = torch.roll(tokens, shifts=(-shift_size, -shift_size), dims=(1, 2)) if shift_size > 0 else tokens  # 按指定距离循环位移探针网格。
    windows = window_partition(shifted, window_size=2)  # 把探针切成二乘二 token 窗口。
    mixed_windows = windows.mean(dim=1, keepdim=True).repeat(1, 4, 1)  # 模拟每个窗口内权重均为四分之一的注意力聚合。
    mixed_grid = window_reverse(mixed_windows, window_size=2, height=4, width=4, batch_size=1)  # 把均匀聚合窗口恢复成二维网格。
    restored = torch.roll(mixed_grid, shifts=(shift_size, shift_size), dims=(1, 2)) if shift_size > 0 else mixed_grid  # 逆位移恢复原空间坐标。
    return restored  # 返回只由窗口拓扑决定的信息传播结果。
probe_tokens = torch.zeros(1, 4, 4, 1)  # 创建全零的四乘四单通道 token 探针。
changed_tokens = probe_tokens.clone()  # 复制探针以只修改一个跨窗相邻位置。
changed_tokens[0, 1, 2, 0] = 4.0  # 把固定窗口右侧的相邻 token 改为四。
regular_before = uniform_window_mix(probe_tokens, shift_size=0)  # 在原探针上模拟固定窗口均匀注意力。
regular_after = uniform_window_mix(changed_tokens, shift_size=0)  # 在修改探针上模拟固定窗口均匀注意力。
shifted_before = uniform_window_mix(probe_tokens, shift_size=1)  # 在原探针上模拟位移窗口均匀注意力。
shifted_after = uniform_window_mix(changed_tokens, shift_size=1)  # 在修改探针上模拟位移窗口均匀注意力。
regular_cross_delta = (regular_after[0, 1, 1] - regular_before[0, 1, 1]).abs().max().item()  # 测量固定窗口下左侧目标位置的变化。
shifted_cross_delta = (shifted_after[0, 1, 1] - shifted_before[0, 1, 1]).abs().max().item()  # 测量位移窗口下目标位置的变化。
print(f"只跑 W-MSA 的跨窗影响={regular_cross_delta:.6f}")  # 展示固定窗口阻断相邻窗口通信。
print(f"再跑 SW-MSA 的跨窗影响={shifted_cross_delta:.6f}")  # 展示位移窗口恢复跨窗信息传播。
print("修复结论：相邻 Block 交替使用 W-MSA 与带边界 mask 的 SW-MSA。")  # 给出结构层面的明确修复方案。

只跑 W-MSA 的跨窗影响=0.000000
再跑 SW-MSA 的跨窗影响=1.000000
修复结论：相邻 Block 交替使用 W-MSA 与带边界 mask 的 SW-MSA。


## 生产差距

真实 Swin 具有多 stage、Patch Merging、随机深度、数据增强和大规模预训练。高分辨率推理还要处理不能整除窗口的 padding、混合精度与 kernel 融合。本实验在全部八张图上拟合，只验证窗口机制，不代表巡检泛化；线上必须用独立设备、时间和货架切分验证。

## 最小回归测试

In [7]:
assert len(sample_names) >= 5  # 保证案例具有足够多的真实可读图像样本。
assert loss_trace[-1] < loss_trace[0]  # 保证真实反向传播使分类损失下降。
assert first_gradient_norm > 0.0  # 保证窗口 QKV 投影获得了非零梯度。
assert swin_accuracy > baseline_accuracy  # 保证手写 Swin 在同数据上超过均值基线。
assert swin_accuracy == 1.0  # 保证教学模型识别了全部八张划痕图。
assert torch.allclose(first_window_attention.sum(dim=1), torch.ones(4), atol=1e-5)  # 保证窗口注意力每行正确归一化。
assert regular_cross_delta < 1e-6 and shifted_cross_delta > 1e-6  # 保证 shifted windows 确实修复跨窗通信。
print("回归测试通过：窗口注意力、训练更新和跨窗连通性均符合预期。")  # 输出集中断言的最终验收结论。

回归测试通过：窗口注意力、训练更新和跨窗连通性均符合预期。
